# CLT-Forge Training Workflow

This notebook follows the official CLT-Forge quick-start workflow end to end:

1. generate and cache activations
2. train a cross-layer transcoder (CLT)
3. run AutoInterp
4. compute an attribution graph
5. launch the visual interface

Target visual style reference:

![CLT-Forge visual interface](https://github.com/LLM-Interp/CLT-Forge/raw/master/images/visual_interface.png)


## Notes

- This notebook is written to mirror the CLT-Forge GitHub instructions closely.
- You will still need a machine with the required GPU memory, PyTorch, and model access for `meta-llama/Llama-3.2-1B`.
- The exact graph you get depends on the trained checkpoint, AutoInterp outputs, and the prompt you trace.
- The final `main(cfg)` cell starts the Dash UI and is expected to block until you stop it.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

CLT_FORGE_REPO = "https://github.com/LLM-Interp/CLT-Forge.git"
CLT_FORGE_DIR = Path("/tmp/CLT-Forge")

if not CLT_FORGE_DIR.exists():
    subprocess.run(["git", "clone", CLT_FORGE_REPO, str(CLT_FORGE_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(CLT_FORGE_DIR)], check=True)

print("Using CLT-Forge from", CLT_FORGE_DIR)


In [ ]:
from pprint import pprint

from clt_forge import (
    ActivationsStore,
    AutoInterp,
    AutoInterpConfig,
    AttributionRunner,
    CLTTrainingRunner,
    clt_training_runner_config,
    load_model,
)
from clt_forge.frontend import AppConfig, main


In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

MODEL_NAME = "meta-llama/Llama-3.2-1B"
DEVICE = "cuda"
RUN_NAME = "llama32_1b_clt_quickstart"
INPUT_STR = 'The opposite of "large" is '

# For repo-specific tracing later, you can replace INPUT_STR with a PowerShell prompt.
# Example:
# INPUT_STR = "powershell -EncodedCommand "

RUN_DIR = PROJECT_ROOT / "artifacts" / "clt_forge" / RUN_NAME
CACHE_DIR = RUN_DIR / "cached_activations"
AUTOINTERP_DIR = RUN_DIR / "autointerp"
GRAPH_DIR = RUN_DIR / "graph"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"

for path in [RUN_DIR, CACHE_DIR, AUTOINTERP_DIR, GRAPH_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Run dir:", RUN_DIR)


In [ ]:
def maybe_set(obj, **kwargs):
    applied = {}
    for key, value in kwargs.items():
        if hasattr(obj, key):
            setattr(obj, key, value)
            applied[key] = value
    return applied


model = load_model(MODEL_NAME, device=DEVICE)
cfg = clt_training_runner_config()

applied = maybe_set(
    cfg,
    model_name=MODEL_NAME,
    device=DEVICE,
    cached_activations_path=str(CACHE_DIR),
    checkpoint_path=str(CHECKPOINT_DIR),
    checkpoints_path=str(CHECKPOINT_DIR),
    output_dir=str(RUN_DIR),
    run_name=RUN_NAME,
)

print("Applied config overrides:")
pprint(applied)

print("\nAvailable config fields:")
pprint(sorted(name for name in dir(cfg) if not name.startswith("_"))[:200])


## Adjust Training Settings If Needed

Inspect the printed config fields above and add any run-specific overrides here before generating activations. In practice you will often want to set cache size, sequence length, dataset source, micro-batch size, expansion factor, or distributed training flags based on your hardware.


In [ ]:
# Example optional overrides. Uncomment and edit the keys that actually exist in your cfg.
# maybe_set(
#     cfg,
#     context_size=256,
#     n_tokens_in_buffer=200000,
#     train_batch_size_tokens=16384,
#     expansion_factor=16,
# )

store = ActivationsStore(model, cfg)
store.generate_and_save_activations(
    path=cfg.cached_activations_path,
    use_compression=True,
)

print("Cached activations written to", cfg.cached_activations_path)


In [ ]:
trainer = CLTTrainingRunner(cfg)
trainer.run()


In [ ]:
def find_latest_checkpoint(root: Path):
    patterns = ["*.pt", "*.pth", "*.ckpt", "*.bin", "*.safetensors"]
    matches = []
    for pattern in patterns:
        matches.extend(root.rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"No checkpoint found under {root}")
    return max(matches, key=lambda path: path.stat().st_mtime)


clt_path = find_latest_checkpoint(CHECKPOINT_DIR if CHECKPOINT_DIR.exists() else RUN_DIR)
print("Using checkpoint:", clt_path)

autointerp_cfg = AutoInterpConfig(
    model_name=MODEL_NAME,
    clt_path=str(clt_path),
)

autointerp = AutoInterp(autointerp_cfg)
autointerp.run(str(AUTOINTERP_DIR))

print("AutoInterp outputs written to", AUTOINTERP_DIR)


In [ ]:
runner = AttributionRunner(
    model_name=MODEL_NAME,
    clt_path=str(clt_path),
)

graph = runner.run(
    input_str=INPUT_STR,
    folder=str(GRAPH_DIR),
)

print("Graph object:", type(graph))
print("Graph outputs written to", GRAPH_DIR)


## Launch The Visual Interface

Run the next cell last. It starts the CLT-Forge Dash app pointed at the attribution graph and AutoInterp outputs produced above.


In [ ]:
app_cfg = AppConfig(
    graph_path=str(GRAPH_DIR),
    autointerp_path=str(AUTOINTERP_DIR),
)

main(app_cfg)
